In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [11]:
df = pd.read_csv('../data/raw/cdc_mental_health.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

C:\Users\jaswa\AppData\Local\Temp\ipykernel_31448\2826266135.py:1: DtypeWarning: Columns (10,11,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/cdc_mental_health.csv')


Shape: (309215, 34)

First 5 rows:


,YearStart,YearEnd,LocationAbbr,LocationDesc,DataSource,Topic,Question,Response,DataValueUnit,DataValueType,...,TopicID,QuestionID,ResponseID,DataValueTypeID,StratificationCategoryID1,StratificationID1,StratificationCategoryID2,StratificationID2,StratificationCategoryID3,StratificationID3
0,2020,2020,US,United States,BRFSS,Health Status,Recent activity limitation among adults,NaN,Number,Age-adjusted Mean,...,HEA,HEA04,NaN,AGEADJMEAN,SEX,SEXF,NaN,NaN,NaN,NaN
1,2015,2019,AR,Arkansas,US Cancer DVT,Cancer,"Invasive cancer (all sites combined), incidence",NaN,Number,Number,...,CAN,CAN07,NaN,NMBR,SEX,SEXM,NaN,NaN,NaN,NaN
2,2015,2019,CA,California,US Cancer DVT,Cancer,"Cervical cancer mortality among all females, u...",NaN,Number,Number,...,CAN,CAN03,NaN,NMBR,OVERALL,OVR,NaN,NaN,NaN,NaN
3,2015,2019,CO,Colorado,US Cancer DVT,Cancer,"Invasive cancer (all sites combined), incidence",NaN,Number,Number,...,CAN,CAN07,NaN,NMBR,RACE,HIS,NaN,NaN,NaN,NaN
4,2015,2019,GA,Georgia,US Cancer DVT,Cancer,"Prostate cancer mortality among all males, und...",NaN,Number,Number,...,CAN,CAN05,NaN,NMBR,RACE,WHT,NaN,NaN,NaN,NaN


In [12]:
print(df.columns.tolist())

['YearStart', 'YearEnd', 'LocationAbbr', 'LocationDesc', 'DataSource', 'Topic', 'Question', 'Response', 'DataValueUnit', 'DataValueType', 'DataValue', 'DataValueAlt', 'DataValueFootnoteSymbol', 'DataValueFootnote', 'LowConfidenceLimit', 'HighConfidenceLimit', 'StratificationCategory1', 'Stratification1', 'StratificationCategory2', 'Stratification2', 'StratificationCategory3', 'Stratification3', 'Geolocation', 'LocationID', 'TopicID', 'QuestionID', 'ResponseID', 'DataValueTypeID', 'StratificationCategoryID1', 'StratificationID1', 'StratificationCategoryID2', 'StratificationID2', 'StratificationCategoryID3', 'StratificationID3']


In [13]:
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

Data Types:
YearStart                      int64
YearEnd                        int64
LocationAbbr                  object
LocationDesc                  object
DataSource                    object
Topic                         object
Question                      object
Response                     float64
DataValueUnit                 object
DataValueType                 object
DataValue                     object
DataValueAlt                  object
DataValueFootnoteSymbol       object
DataValueFootnote             object
LowConfidenceLimit            object
HighConfidenceLimit           object
StratificationCategory1       object
Stratification1               object
StratificationCategory2      float64
Stratification2              float64
StratificationCategory3      float64
Stratification3              float64
Geolocation                   object
LocationID                     int64
TopicID                       object
QuestionID                    object
ResponseID                

In [14]:
# Check what topics are available
print(df['Topic'].value_counts())

Topic
Cardiovascular Disease                             30709
Chronic Obstructive Pulmonary Disease              26951
Nutrition, Physical Activity, and Weight Status    26069
Health Status                                      25612
Alcohol                                            25321
Cancer                                             22358
Immunization                                       19140
Social Determinants of Health                      19026
Arthritis                                          18645
Diabetes                                           17318
Mental Health                                      16976
Tobacco                                            15931
Oral Health                                        13236
Asthma                                             10511
Cognitive Health and Caregiving                     9904
Sleep                                               5318
Disability                                          5060
Maternal Health          

In [15]:
# Filter for mental health topic only
mental_health_df = df[df['Topic'] == 'Mental Health']

print("Mental Health rows:", mental_health_df.shape)
print("\nQuestions available:")
print(mental_health_df['Question'].value_counts())

Mental Health rows: (16976, 34)

Questions available:
Question
Frequent mental distress among adults                                  5060
Depression among adults                                                5060
Average mentally unhealthy days among adults                           5060
Postpartum depressive symptoms among women with a recent live birth    1026
Current poor mental health among high school students                   770
Name: count, dtype: int64


In [16]:
# Keep only the columns we need
columns_to_keep = [
    'YearStart', 'YearEnd', 'LocationAbbr', 'LocationDesc',
    'Topic', 'Question', 'DataValue', 'DataValueUnit',
    'DataValueType', 'StratificationCategory1', 'Stratification1',
    'StratificationCategory2', 'Stratification2'
]

mental_health_df = mental_health_df[columns_to_keep]
print("Shape after dropping columns:", mental_health_df.shape)

Shape after dropping columns: (16976, 13)


In [17]:
# Check nulls in our filtered dataset
print("Nulls before cleaning:")
print(mental_health_df.isnull().sum())

# Drop rows where DataValue is null (can't analyze without values)
mental_health_df = mental_health_df.dropna(subset=['DataValue'])

# Fill remaining nulls
mental_health_df['Stratification1'] = mental_health_df['Stratification1'].fillna('Overall')
mental_health_df['Stratification2'] = mental_health_df['Stratification2'].fillna('Overall')

print("\nNulls after cleaning:")
print(mental_health_df.isnull().sum())

Nulls before cleaning:
YearStart                      0
YearEnd                        0
LocationAbbr                   0
LocationDesc                   0
Topic                          0
Question                       0
DataValue                   5244
DataValueUnit                  0
DataValueType                  0
StratificationCategory1        0
Stratification1                0
StratificationCategory2    16976
Stratification2            16976
dtype: int64

Nulls after cleaning:
YearStart                      0
YearEnd                        0
LocationAbbr                   0
LocationDesc                   0
Topic                          0
Question                       0
DataValue                      0
DataValueUnit                  0
DataValueType                  0
StratificationCategory1        0
Stratification1                0
StratificationCategory2    11732
Stratification2                0
dtype: int64


In [18]:
# Remove national-level rows, keep only states
mental_health_df = mental_health_df[mental_health_df['LocationAbbr'] != 'US']

# Filter years 2018-2023 (pre vs post COVID)
mental_health_df = mental_health_df[mental_health_df['YearStart'].between(2018, 2023)]

# Convert DataValue to numeric just in case
mental_health_df['DataValue'] = pd.to_numeric(mental_health_df['DataValue'], errors='coerce')

# Reset index
mental_health_df = mental_health_df.reset_index(drop=True)

print("Final shape:", mental_health_df.shape)
print("\nYear range:", mental_health_df['YearStart'].min(), "-", mental_health_df['YearStart'].max())
print("\nNumber of states:", mental_health_df['LocationAbbr'].nunique())
print("\nSample:")
mental_health_df.head()

Final shape: (11479, 13)

Year range: 2019 - 2022

Number of states: 54

Sample:


,YearStart,YearEnd,LocationAbbr,LocationDesc,Topic,Question,DataValue,DataValueUnit,DataValueType,StratificationCategory1,Stratification1,StratificationCategory2,Stratification2
0,2019,2019,AL,Alabama,Mental Health,Frequent mental distress among adults,21.0,%,Crude Prevalence,Age,Age 18-44,NaN,Overall
1,2019,2019,AR,Arkansas,Mental Health,Depression among adults,27.8,%,Crude Prevalence,Age,Age 18-44,NaN,Overall
2,2019,2019,GU,Guam,Mental Health,Depression among adults,13.0,%,Age-adjusted Prevalence,Race/Ethnicity,"Multiracial, non-Hispanic",NaN,Overall
3,2019,2019,ID,Idaho,Mental Health,Depression among adults,22.3,%,Crude Prevalence,Age,Age 45-64,NaN,Overall
4,2019,2019,ID,Idaho,Mental Health,Frequent mental distress among adults,11.5,%,Crude Prevalence,Sex,Male,NaN,Overall
